# {{PROJECT_NAME}}

KFP project on the Miramar platform.

- Define pipeline components in the tagged cells below (`kfp_step`, `kfp_pipeline`)
- Run the **Build → `pipeline.py`** cell to regenerate `pipeline.py`
- Use the Compile & Submit cells to test runs locally
- Use **Deploy to KFP** (GitHub Actions) for CI/CD submission

## Pipeline definition

Define your components below. Tag each component cell `kfp_step` and the pipeline cell `kfp_pipeline`.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=["nvtx"],
)
def gpu_stage(run_id: str):
    import torch
    import nvtx

    # For Nsight profiling (config.yaml profiling.enabled): the GPU must be hot the
    # instant collection opens. Issue real GPU kernels from the FIRST line below - no
    # startup sleep, no data-load idle - and keep the GPU busy for the whole
    # collection_window_s. GB10 hw-trace only timestamps kernels from the first few
    # seconds after collection opens; a slow start yields a kernel-less report.
    # ── Replace this block with your GPU workload ─────────────────────────
    with nvtx.annotate("gpu_stage", color="green"):
        x = torch.randn(2048, 2048, device="cuda")
        result = x @ x.T
        torch.cuda.synchronize()

    print(f"Result norm: {result.norm().item():.4f}")
    # ─────────────────────────────────────────────────────────────────────

In [ ]:
import yaml as _yaml, pathlib as _pathlib

_cfg = _yaml.safe_load(_pathlib.Path("config.yaml").read_text()) or {}
_profiling_cfg = _cfg.get("profiling", {})


@dsl.pipeline(
    name="{{PROJECT_NAME}}",
    description="GPU pipeline — replace gpu_stage with your workload.",
)
def pipeline(run_id: str = "run-001"):
    task = gpu_stage(run_id=run_id)
    task.set_gpu_limit(1).set_memory_limit("64G")

    # Nsight Operator GPU profiling — opt-in via config.yaml profiling.enabled. A true
    # value labels this stage pod nvidia-nsight-profile=enabled so the operator webhook
    # injects nsys capture; then run /nsight-export <project> <run-NNN> main while the pod
    # is GPU-hot to archive the report into ~/shared/nsight/<project>/<run-id>/main/.
    # Never label the kubeflow namespace itself.
    if _profiling_cfg.get("enabled", False):
        kubernetes.add_pod_label(task, label_key="nvidia-nsight-profile", label_value="enabled")

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell to regenerate `pipeline.py`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/build_pipeline.py"], check=True)

## Compile & Submit

Run these cells to compile and submit directly from Jupyter (useful for quick iteration).

In [ ]:
import kfp
from kfp import compiler, dsl
print(f'kfp {kfp.__version__}')

In [ ]:
# Compile the pipeline defined in pipeline.py
from pipeline import pipeline
compiler.Compiler().compile(pipeline_func=pipeline, package_path='/tmp/pipeline.yaml')
print('Compiled → /tmp/pipeline.yaml')

In [ ]:
# Connect to KFP (requires SSH tunnel: ssh -L 8080:localhost:8080 spark-79b7.local)
client = kfp.Client(host='http://localhost:8080')
client.list_pipelines()

In [ ]:
# Submit a run
run = client.create_run_from_pipeline_package(
    pipeline_file='/tmp/pipeline.yaml',
    arguments={},
    run_name='notebook-run',
)
print(f'Run ID: {run.run_id}')
print(f'UI: http://localhost:8080/#/runs/details/{run.run_id}')

In [ ]:
# Poll run status
import time
run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f'  {state}')
    if state in ('SUCCEEDED', 'FAILED', 'CANCELED'):
        break
    time.sleep(10)